# 15 · Long-Context Forecasting

TimesFM 2.5 supports up to **16,384** context points — 8× the previous version.
Long context lets the model learn multiple nested seasonalities (hour, day,
week, year). Here we raise `max_context` and feed a long high-frequency series.

In [ ]:
import numpy as np, torch, timesfm
torch.set_float32_matmul_precision("high")
model = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
model.compile(timesfm.ForecastConfig(
    max_context=4096,           # long window (raise toward 16384 if RAM allows)
    max_horizon=336,
    normalize_inputs=True, use_continuous_quantile_head=True,
    fix_quantile_crossing=True,
))

In [ ]:
# ~85 days of hourly data (2040 points) with daily + weekly + slow trend
rng = np.random.default_rng(8)
h = np.arange(2040)
series = (
    500 + 0.02*h
    + 120*np.sin(2*np.pi*(h % 24)/24)
    + 60*np.sin(2*np.pi*h/(24*7))
    + rng.normal(0, 15, h.size)
).astype(np.float32)
print("context points:", series.size)

In [ ]:
horizon = 168   # one week ahead, hourly
point, q = model.forecast(horizon=horizon, inputs=[series])
point, q = point[0], q[0]
print("forecast horizon:", point.size, "hours")

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
hist = series[-24*10:]
xf = range(len(hist), len(hist)+horizon)
fig, ax = plt.subplots(figsize=(15,5))
ax.plot(range(len(hist)), hist, color="tab:blue", lw=0.8, label="last 10 days")
ax.plot(xf, point, color="tab:purple", lw=1.2, label="1-week forecast")
ax.fill_between(xf, q[:,1], q[:,9], color="tab:purple", alpha=0.2)
ax.set_title("Long-context hourly forecast (1 week ahead)"); ax.legend()
fig.tight_layout(); fig.savefig("long_context.png", dpi=130)
print("saved long_context.png")

### Trade-off
Longer context and horizon cost more memory and time. Use the preflight
estimator (notebook 01) to size your machine:

```bash
python ../timesfm-forecasting/scripts/check_system.py \
    --num-series 1 --context-length 4096 --horizon 336 --estimate-only
```